# Laya fine-tune: Hyperliquid states (small trial)Baseline (pretrained) vs fine-tuned accuracy on our labeled val set.Training: official Laya RLCD script on 2xT4 (falls back to 1 GPU).

## 1. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("cuda:", torch.cuda.is_available(), "| gpus:", torch.cuda.device_count())


## 2. Install deps

In [ ]:
!pip install -q -U "laya>=0.3.21" "transformers>=4.48.0" "datasets>=3.0.0" safetensors huggingface_hub pyarrow pandas scikit-learn


## 3. Download model + preprocess our data

In [ ]:
import os, json, torch
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
from laya.agent import _fix_tokenizer_config

MODEL_ID = "convaiinnovations/laya-multilingual"

# Dataset attached as Kaggle input: kinan21/laya-hl-trial-v1
DATA_DIR = None
for cand in ["/kaggle/input/laya-hl-trial-v1",
             "/kaggle/input/datasets/kinan21/laya-hl-trial-v1"]:
    if os.path.exists(os.path.join(cand, "train.jsonl")):
        DATA_DIR = cand
        break
assert DATA_DIR, "Dataset kinan21/laya-hl-trial-v1 not mounted. Attach it via Add Input."
print("DATA_DIR =", DATA_DIR)
print(f"Fetching model from {MODEL_ID} (~1 min on Kaggle's network)...")
model_dir = snapshot_download(MODEL_ID)
_fix_tokenizer_config(model_dir)

tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
    cfg = json.load(f)

def build_training_item(state, q, gold_q):
    t = q["type"]
    crit = q.get("criteria", {})
    if t == "choice":
        keys = list(crit.keys())
        target = [gold_q["probabilities"].get(k, 0.0) for k in keys]
    elif t == "noul":
        target = [gold_q["probabilities"].get("false", 0.5), gold_q["probabilities"].get("true", 0.5)]
    elif t == "score":
        n_levels = len(crit) if isinstance(crit, list) else 4
        target = [gold_q["probabilities"].get(str(i), 0.0) for i in range(n_levels)]
    s = sum(target)
    target = [v / s for v in target] if s > 0 else [1.0 / len(target)] * len(target)
    label = target.index(max(target))
    k = len(render_options({"t": t, "crit": crit}))
    seq, markers = build_sequence(tok, state, {"t": t, "ins": q["instructions"], "crit": crit}, cfg["max_len"], cfg["head_max_len"])
    if len(markers) != k:
        return None
    return {"ids": seq, "markers": markers, "qtype": QTYPES[t], "target": target, "label": label}

def load_rows(path):
    rows = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

for split in ["train", "val"]:
    rows = load_rows(os.path.join(DATA_DIR, f"{split}.jsonl"))
    items, dropped = [], 0
    for row in rows:
        # NOTE: our state is already rendered text (laya-live prompt), NOT a JSON dict
        state = row["state"] if isinstance(row["state"], str) else json.dumps(row["state"])
        questions = json.loads(row["questions"]) if isinstance(row["questions"], str) else row["questions"]
        gold = json.loads(row["gold"]) if isinstance(row["gold"], str) else row["gold"]
        for qid, q in questions.items():
            if qid in gold:
                it = build_training_item(state, q, gold[qid])
                if it:
                    items.append(it)
                else:
                    dropped += 1
    torch.save(items, f"/kaggle/working/{split}_items.pt")
    print(f"{split}: {len(rows)} cases -> {len(items)} items (dropped {dropped})")

with open("/kaggle/working/model_dir.txt", "w") as f:
    f.write(model_dir)
print("model_dir:", model_dir)



## 4. Training script (official Laya RLCD, unmodified)

In [ ]:
%%writefile /kaggle/working/train_ddp.py
import os, sys, time, json, random, math
import numpy as np
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from safetensors.torch import load_file, save_file
from transformers import AutoTokenizer
from laya.common import build_model, proper_reward, QTYPES

def collate_train_batch(items, pad_id):
    n, L = len(items), max(len(it["ids"]) for it in items)
    kmax = max(len(it["markers"]) for it in items)
    ids = torch.full((n, L), pad_id, dtype=torch.long)
    att = torch.zeros((n, L), dtype=torch.long)
    mpos = torch.zeros((n, kmax), dtype=torch.long)
    mmask = torch.zeros((n, kmax), dtype=torch.bool)
    target = torch.zeros((n, kmax), dtype=torch.float32)
    for i, it in enumerate(items):
        ids[i, : len(it["ids"])] = torch.tensor(it["ids"])
        att[i, : len(it["ids"])] = 1
        k = len(it["markers"])
        mpos[i, :k] = torch.tensor(it["markers"])
        mmask[i, :k] = True
        target[i, : len(it["target"])] = torch.tensor(it["target"], dtype=torch.float32)
    return {
        "input_ids": ids,
        "attention_mask": att,
        "marker_pos": mpos,
        "marker_mask": mmask,
        "target": target,
        "qtype": torch.tensor([it["qtype"] for it in items]),
        "label": torch.tensor([it["label"] for it in items])
    }

def fit_one_temp(sel):
    if len(sel) < 10:
        return 1.0
    kmax = max(len(z) for z, _ in sel)
    Z = torch.full((len(sel), kmax), -1e4)
    T = torch.zeros((len(sel), kmax))
    for i, (z, t) in enumerate(sel):
        Z[i, :len(z)] = torch.tensor(z)
        T[i, :len(t)] = torch.tensor(t, dtype=torch.float32)
    log_t = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100)
    def closure():
        opt.zero_grad()
        loss = -(T * torch.log_softmax(Z / log_t.exp(), -1)).sum(-1).mean()
        loss.backward()
        return loss
    opt.step(closure)
    return float(torch.clamp(log_t.exp(), 0.1, 10.0).item())

def main():
    dist.init_process_group("nccl")
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank)

    model_dir = sys.argv[1]
    output_dir = sys.argv[2]
    
    with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
        cfg = json.load(f)
    cfg["gradient_checkpointing"] = True
    cfg["max_tokens_per_batch"] = 4096
    cfg["max_len"] = 1024
    cfg["head_max_len"] = 256

    tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
    model = build_model(cfg, encoder_dir=os.path.join(model_dir, "encoder"))
    
    weights = load_file(os.path.join(model_dir, "model.safetensors"))
    model.load_state_dict(weights, strict=True)
    
    model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.head_checkpointing = True
    model.to(device)
    model.train()

    ddp_model = DDP(model, device_ids=[local_rank], find_unused_parameters=True)
    
    all_items = torch.load("/kaggle/working/train_items.pt", weights_only=False)

    # Hold the calibration slice out of training before sharding. Temperatures fitted on
    # items the run has already trained on measure the fit rather than the calibration: the
    # model is near-certain and near-correct on them, so the optimiser has nothing to soften
    # and returns a degenerate scale. The seed is fixed and rank-independent, so every rank
    # withholds exactly the same items and none of them reaches a training batch.
    CALIB_MAX = 400
    order = list(range(len(all_items)))
    random.Random(20260922).shuffle(order)
    n_calib = min(CALIB_MAX, len(all_items) // 10)
    calib_items = [all_items[i] for i in sorted(order[:n_calib])]
    train_items = [all_items[i] for i in sorted(order[n_calib:])]
    my_items = train_items[rank::world_size]
    
    EPOCHS = 4
    MICRO_BATCH = 8      # 8 sequences per forward pass per GPU
    GRAD_ACCUM = 4       # Effective batch across 2 GPUs = 64 sequences (8 * 2 * 4)
    GROUP_SIZE = 4       # GRPO baseline samples
    LR_ENCODER = 2.5e-5  # Encoder adaptation rate
    LR_HEAD = 1.0e-4     # Head adaptation rate
    SIGMA_START = 0.4    # Exploration noise
    SIGMA_END = 0.1

    enc_params = [p for n, p in ddp_model.named_parameters() if "encoder." in n]
    head_params = [p for n, p in ddp_model.named_parameters() if "encoder." not in n]
    
    optimizer = torch.optim.AdamW([
        {"params": enc_params, "lr": LR_ENCODER},
        {"params": head_params, "lr": LR_HEAD}
    ], weight_decay=0.01)
    
    total_updates = (len(my_items) // (MICRO_BATCH * GRAD_ACCUM)) * EPOCHS
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, total_updates), eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda", enabled=True)
    
    if rank == 0:
        print(f"Starting 2xT4 DDP training: {len(train_items)} train items ({len(calib_items)} held out for calibration) | {len(my_items)} per rank | {EPOCHS} epochs")
    t0 = time.time()
    
    for epoch in range(EPOCHS):
        random.seed(42 + epoch + rank)
        random.shuffle(my_items)
        epoch_loss, n_batches = 0.0, 0
        optimizer.zero_grad(set_to_none=True)
        accum_step = 0
        
        progress = epoch / max(1, EPOCHS - 1)
        sigma = SIGMA_START + (SIGMA_END - SIGMA_START) * progress
        
        for b_idx in range(0, len(my_items), MICRO_BATCH):
            chunk = my_items[b_idx:b_idx + MICRO_BATCH]
            if not chunk:
                continue
            
            batch = collate_train_batch(chunk, tok.pad_token_id)
            
            with torch.autocast("cuda", dtype=torch.float16):
                logits, act = ddp_model(
                    batch["input_ids"].to(device),
                    batch["attention_mask"].to(device),
                    batch["marker_pos"].to(device),
                    batch["marker_mask"].to(device),
                    batch["qtype"].to(device)
                )
            
            logits = logits.float()
            mask = batch["marker_mask"].to(device)
            k = mask.sum(-1, keepdim=True).float()
            target = batch["target"].to(device)
            
            # 1. Sample G noisy logit distributions with zero-mean projection
            eps = torch.randn((GROUP_SIZE,) + logits.shape, device=device) * sigma * mask
            eps = (eps - eps.sum(-1, keepdim=True) / k) * mask
            z = logits.detach().unsqueeze(0) + eps
            q = torch.softmax(z.masked_fill(~mask, -1e4), -1)
            
            # 2. Evaluate proper scoring reward (w_sph=0.75 for soft target matching)
            with torch.no_grad():
                r = proper_reward(q, target.unsqueeze(0), batch["qtype"].to(device), mask, w_sph=0.75, w_rps=1.0)
                adv = r - r.mean(0, keepdim=True)
                adv = adv / (adv.std() + 1e-6)
            
            # 3. Policy gradient loss + full 1.0 soft cross-entropy guidance
            logp = -(((z - logits.unsqueeze(0)) ** 2) * mask).sum(-1) / (2 * sigma ** 2)
            loss_rl = -(adv * logp).mean()
            loss_ce = -(target * torch.log_softmax(logits.masked_fill(~mask, -1e4), -1)).sum(-1).mean()
            loss = (loss_rl + 1.0 * loss_ce) / GRAD_ACCUM + 0.0 * act.sum()
            
            scaler.scale(loss).backward()
            accum_step += 1
            
            if accum_step % GRAD_ACCUM == 0 or (b_idx + MICRO_BATCH) >= len(my_items):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(ddp_model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
            
            epoch_loss += loss.item() * GRAD_ACCUM
            n_batches += 1
            
            if rank == 0 and (n_batches % 50) == 0:
                cur_lr = scheduler.get_last_lr()[0]
                print(f"  Epoch {epoch+1}/{EPOCHS} | Step {n_batches} | Loss: {loss.item()*GRAD_ACCUM:.4f} | Reward: {r.mean().item():.3f} | LR: {cur_lr:.2e}")

        if rank == 0:
            print(f"=== Epoch {epoch+1}/{EPOCHS} Completed in {time.time()-t0:.1f}s | Avg Loss: {epoch_loss/max(1, n_batches):.4f} ===")

        dist.barrier()

        # Overwrite a single rolling checkpoint after each epoch so a crash,
        # OOM, or Kaggle session timeout doesn't lose all prior training.
        if rank == 0:
            ckpt_dir = os.path.join(output_dir, "checkpoint_latest")
            os.makedirs(ckpt_dir, exist_ok=True)
            ckpt_sd = {k: v.half().contiguous().cpu() for k, v in model.state_dict().items()}
            save_file(ckpt_sd, os.path.join(ckpt_dir, "model.safetensors"))
            model.encoder.config.save_pretrained(os.path.join(ckpt_dir, "encoder"))
            tok.save_pretrained(os.path.join(ckpt_dir, "tokenizer"))
            with open(os.path.join(ckpt_dir, "checkpoint_meta.json"), "w") as f:
                json.dump({
                    "epoch": epoch + 1,
                    "total_epochs": EPOCHS,
                    "avg_loss": epoch_loss / max(1, n_batches)
                }, f, indent=2)
            print(f"  Saved rolling checkpoint (epoch {epoch+1}/{EPOCHS}) to {ckpt_dir}")

    dist.barrier()
    
    # Post-training temperature calibration on rank 0 (micro-batched in chunks of 16 to prevent OOM)
    if rank == 0:
        print("\nFitting post-training calibration temperatures...")
        del optimizer, scaler, scheduler
        torch.cuda.empty_cache()
        model.eval()
        # calib_items was held out above and never entered a training batch
        calib_preds = []
        with torch.no_grad():
            for c_idx in range(0, len(calib_items), 16):
                c_chunk = calib_items[c_idx:c_idx + 16]
                cb = collate_train_batch(c_chunk, tok.pad_token_id)
                with torch.autocast("cuda", dtype=torch.float16):
                    l_sub, _ = model(
                        cb["input_ids"].to(device),
                        cb["attention_mask"].to(device),
                        cb["marker_pos"].to(device),
                        cb["marker_mask"].to(device),
                        cb["qtype"].to(device)
                    )
                l_np = l_sub.float().cpu().numpy()
                for r, it in enumerate(c_chunk):
                    k = len(it["markers"])
                    calib_preds.append((it["qtype"], l_np[r, :k], it["target"]))
        
        fitted_temps = [1.2, 1.2, 1.2]
        try:
            for qt in range(3):
                sel = [(z, t) for q_type, z, t in calib_preds if q_type == qt]
                if sel:
                    fitted_temps[qt] = fit_one_temp(sel)
            print("Fitted calibration temperatures (choice, score, noul):", [round(t, 3) for t in fitted_temps])
        except Exception as e:
            print("Temperature fitting fallback:", e)
        os.makedirs(output_dir, exist_ok=True)
        sd = {k: v.half().contiguous().cpu() for k, v in model.state_dict().items()}
        save_file(sd, os.path.join(output_dir, "model.safetensors"))
        model.encoder.config.save_pretrained(os.path.join(output_dir, "encoder"))
        tok.save_pretrained(os.path.join(output_dir, "tokenizer"))
        
        cfg["fine_tuned"] = True
        cfg["model_name"] = "laya-typed-decisions"
        cfg["temperature"] = fitted_temps
        # This fit is per type; inherited bucket overrides would hide the new values.
        cfg.pop("temperature_by_options", None)
        with open(os.path.join(output_dir, "rl_agent_config.json"), "w") as f:
            json.dump(cfg, f, indent=2)
        print(f"Model successfully saved to {output_dir}!")

    dist.destroy_process_group()

if __name__ == "__main__":
    main()



## 5. Launch training

In [ ]:
import os, subprocess, torch
n_gpu = torch.cuda.device_count()
print(f"GPUs available: {n_gpu}")
model_dir = open("/kaggle/working/model_dir.txt").read().strip()
OUTPUT_DIR = "/kaggle/working/laya_hl_small"
os.makedirs(OUTPUT_DIR, exist_ok=True)
nproc = max(1, n_gpu)
cmd = (
    f"torchrun --standalone --nproc_per_node={nproc} /kaggle/working/train_ddp.py "
    f"{model_dir} {OUTPUT_DIR}"
)
print("Launching:", cmd)
ret = subprocess.run(cmd, shell=True)
print("torchrun exit code:", ret.returncode)
assert ret.returncode == 0, "training failed"



## 6. Eval: baseline vs finetuned on our val set

In [ ]:
import time, json, os
import numpy as np
import laya

DATA_DIR = resolve_data_dir()
assert DATA_DIR, "dataset not mounted"
print("DATA_DIR =", DATA_DIR)
model_dir = open("/kaggle/working/model_dir.txt").read().strip()
OUTPUT_DIR = "/kaggle/working/laya_hl_small"

val_rows = []
with open(os.path.join(DATA_DIR, "val.jsonl")) as f:
    for line in f:
        if line.strip():
            val_rows.append(json.loads(line))
print(f"val cases: {len(val_rows)}")

def evaluate(agent, rows, tag):
    acc = {}
    brier = {}
    lat = []
    n_q = 0
    for row in val_rows:
        state = row["state"] if isinstance(row["state"], str) else json.dumps(row["state"])
        questions = json.loads(row["questions"]) if isinstance(row["questions"], str) else row["questions"]
        gold = json.loads(row["gold"]) if isinstance(row["gold"], str) else row["gold"]
        t0 = time.perf_counter()
        res = agent.predict(state, questions)
        lat.append((time.perf_counter() - t0) * 1000)
        for qid, qdef in questions.items():
            keys = list(qdef["criteria"].keys())
            pred_choice = res["answers"][qid]["choice"]
            is_corr = float(pred_choice == str(gold[qid]["label"]))
            p = np.array([res["answers"][qid]["probabilities"].get(k, 1e-6) for k in keys])
            p /= p.sum()
            g = np.zeros(len(keys)); g[keys.index(str(gold[qid]["label"]))] = 1.0
            acc.setdefault(qid, []).append(is_corr)
            brier.setdefault(qid, []).append(float(((p - g) ** 2).sum()))
            n_q += 1
    out = {qid: float(np.mean(v)) for qid, v in acc.items()}
    out["overall_acc"] = float(np.mean([v for vs in acc.values() for v in vs]))
    out["overall_brier"] = float(np.mean([v for vs in brier.values() for v in vs]))
    out["mean_latency_ms"] = float(np.mean(lat))
    out["n_cases"] = len(rows)
    print(f"[{tag}] " + " | ".join(f"{k}={v:.3f}" for k, v in out.items() if k != "n_cases"))
    return out

print("== baseline (pretrained) ==")
agent_base = laya.Agent(model_dir, device="cuda")
m_base = evaluate(agent_base, val_rows, "baseline")
del agent_base

print("== finetuned ==")
agent_ft = laya.Agent(OUTPUT_DIR, device="cuda")
m_ft = evaluate(agent_ft, val_rows, "finetuned")
del agent_ft

report = {"baseline": m_base, "finetuned": m_ft,
          "val_cases": len(val_rows),
          "note": "accuracy vs gold labels on chronological val split (last 20% of window)"}
with open("/kaggle/working/eval_report.json", "w") as f:
    json.dump(report, f, indent=2)

print()
print(f"{'question':10} {'baseline':>9} {'finetuned':>9} {'delta':>8}")
for qid in m_base:
    if qid in ("overall_acc", "overall_brier", "mean_latency_ms", "n_cases"):
        continue
    b, f_ = m_base[qid], m_ft[qid]
    print(f"{qid:10} {b:>9.3f} {f_:>9.3f} {f_-b:>+8.3f}")
print(f"{'OVERALL':10} {m_base['overall_acc']:>9.3f} {m_ft['overall_acc']:>9.3f} {m_ft['overall_acc']-m_base['overall_acc']:>+8.3f}")
print(f"Brier: baseline {m_base['overall_brier']:.4f} vs finetuned {m_ft['overall_brier']:.4f} (lower is better)")
print(f"Latency: baseline {m_base['mean_latency_ms']:.0f}ms vs finetuned {m_ft['mean_latency_ms']:.0f}ms per 3-question decision")
print("report saved to /kaggle/working/eval_report.json")

